In [0]:
from pyspark.sql import functions as F

# ============================================================
# 1. Load Silver and Gold
# ============================================================

silver_df = spark.table(
    "customer360_dev.silver.customers"
)

gold_df = spark.table(
    "customer360_dev.gold.dim_customer"
)

In [0]:
multiple_current_count = (
    gold_df
    .filter(F.col("is_current") == True)
    .groupBy("customer_id")
    .count()
    .filter(F.col("count") > 1)
    .count()
)

print(
    "Customers with multiple current versions:",
    multiple_current_count
)

Customers with multiple current versions: 0


In [0]:
missing_current_gold = (
    silver_df
    .select("customer_id")
    .join(
        gold_df
        .filter(F.col("is_current") == True)
        .select("customer_id"),

        on="customer_id",
        how="left_anti"
    )
    .count()
)

print(
    "Silver customers missing current Gold version:",
    missing_current_gold
)

Silver customers missing current Gold version: 0


In [0]:
orphan_current_gold = (
    gold_df
    .filter(F.col("is_current") == True)
    .select("customer_id")
    .join(
        silver_df.select("customer_id"),
        on="customer_id",
        how="left_anti"
    )
    .count()
)

print(
    "Current Gold customers missing in Silver:",
    orphan_current_gold
)

Current Gold customers missing in Silver: 0


In [0]:
invalid_date_ranges = (
    gold_df
    .filter(
        F.col("effective_from").isNull()
        |
        F.col("effective_to").isNull()
        |
        (
            F.col("effective_from")
            >
            F.col("effective_to")
        )
    )
    .count()
)

print(
    "Invalid SCD2 date ranges:",
    invalid_date_ranges
)

Invalid SCD2 date ranges: 0


In [0]:
invalid_current_end_dates = (
    gold_df
    .filter(F.col("is_current") == True)
    .filter(
        F.col("effective_to")
        !=
        F.to_timestamp(
            F.lit("9999-12-31 23:59:59")
        )
    )
    .count()
)

print(
    "Current rows with invalid effective_to:",
    invalid_current_end_dates
)

Current rows with invalid effective_to: 0


In [0]:
invalid_historical_rows = (
    gold_df
    .filter(F.col("is_current") == False)
    .filter(
        F.col("effective_to")
        ==
        F.to_timestamp(
            F.lit("9999-12-31 23:59:59")
        )
    )
    .count()
)

print(
    "Historical rows still open-ended:",
    invalid_historical_rows
)

Historical rows still open-ended: 0


In [0]:
current_gold_df = (
    gold_df
    .filter(F.col("is_current") == True)
)

silver_gold_compare_df = (
    silver_df.alias("s")
    .join(
        current_gold_df.alias("g"),
        F.col("s.customer_id")
        ==
        F.col("g.customer_id"),
        "inner"
    )
)

In [0]:
current_state_mismatch = (
    silver_gold_compare_df
    .filter(

        ~F.col("s.first_name").eqNullSafe(
            F.col("g.first_name")
        )

        |

        ~F.col("s.last_name").eqNullSafe(
            F.col("g.last_name")
        )

        |

        ~F.col("s.email").eqNullSafe(
            F.col("g.email")
        )

        |

        ~F.col("s.phone_number").eqNullSafe(
            F.col("g.phone_number")
        )

        |

        ~F.col("s.date_of_birth").eqNullSafe(
            F.col("g.date_of_birth")
        )

        |

        ~F.col("s.gender").eqNullSafe(
            F.col("g.gender")
        )

        |

        ~F.col("s.customer_status").eqNullSafe(
            F.col("g.customer_status")
        )

        |

        ~F.col("s.loyalty_tier").eqNullSafe(
            F.col("g.loyalty_tier")
        )

        |

        ~F.col("s.preferred_language").eqNullSafe(
            F.col("g.preferred_language")
        )
    )
    .count()
)

print(
    "Silver vs current Gold mismatches:",
    current_state_mismatch
)

Silver vs current Gold mismatches: 0


In [0]:
reconciliation_pass = (

    multiple_current_count == 0

    and missing_current_gold == 0

    and orphan_current_gold == 0

    and invalid_date_ranges == 0

    and invalid_current_end_dates == 0

    and invalid_historical_rows == 0

    and current_state_mismatch == 0

)

status = (
    "PASS"
    if reconciliation_pass
    else "FAIL"
)

print(
    "Customer SCD2 Reconciliation:",
    status
)

Customer SCD2 Reconciliation: PASS


In [0]:
result = [
    {
        "dimension": "dim_customer",
        "multiple_current_rows": multiple_current_count,
        "missing_current_gold": missing_current_gold,
        "orphan_current_gold": orphan_current_gold,
        "invalid_date_ranges": invalid_date_ranges,
        "invalid_current_end_dates": invalid_current_end_dates,
        "invalid_historical_rows": invalid_historical_rows,
        "current_state_mismatch": current_state_mismatch,
        "status": status
    }
]

display(
    spark.createDataFrame(result)
)

current_state_mismatch,dimension,invalid_current_end_dates,invalid_date_ranges,invalid_historical_rows,missing_current_gold,multiple_current_rows,orphan_current_gold,status
0,dim_customer,0,0,0,0,0,0,PASS
